# Reproduce the results

This notebook regenerates every reported score **without retraining and without
downloading the dataset to a local computer**. It reads the converted dataset
from Google Drive, unpacks it on Colab's local disk, runs the repository's
scripts, and saves only the small result files (a few hundred KB) back to Drive.

| Step | What it runs | Writes |
|---|---|---|
| B | `scripts/05_export_sequences.py` | `results/split_manifest.csv`, `results/split_check.json`, the 30 road sequences |
| C1 | `scripts/08_evaluate_models.py` | `results/model_metrics.json` (detector mAP, classifier top-1) |
| C2-C3 | `scripts/07_batch_evaluate.py`, with and without the crop classifier | `results/batch_evaluation_stage2.json`, `..._stage1.json`, `weights/calibration.json` |
| D3 | `scripts/11_sequence_checks.py` | `results/sequence_checks.json` (baselines and sanity checks) |
| E | compare and save | `export/reproduced_results.zip` on Drive |

**Required on Google Drive**
- `traffic-sign-inventory/datasets/arts_yolo.zip`, the converted dataset made by
  `01_train_yolo_colab.ipynb` (phases A7 to C4). The dataset itself is not
  redistributed with the repository.
- Nothing else: cell A3 clones the code from GitHub. To use a local copy instead,
  upload a zip of the repository to `traffic-sign-inventory/repo_for_colab.zip`
  and set `REPO_URL = ""` in cell A2.

**Runtime:** a T4 GPU is recommended (*Runtime > Change runtime type > T4 GPU*);
a CPU runtime also works, much more slowly. Copying the dataset in B1 takes about
3 minutes.

## A. Setup

In [1]:
# A1. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# A2. Settings. Edit DRIVE_ROOT if the folder lives elsewhere.
DRIVE_ROOT  = "/content/drive/MyDrive/traffic-sign-inventory"
DATASET_ZIP = f"{DRIVE_ROOT}/datasets/arts_yolo.zip"       # from notebook 01, cell C4

# Where the code comes from: a GitHub URL and branch, or a zip of the repository on Drive.
REPO_URL    = "https://github.com/JawadGigyani/Geo-Localization-of-Traffic-Signs.git"   # "" to use REPO_ZIP
REPO_BRANCH = "main"
REPO_ZIP    = f"{DRIVE_ROOT}/repo_for_colab.zip"

REPO        = "/content/repo"
RESULTS_ZIP = f"{DRIVE_ROOT}/export/reproduced_results.zip"

import os
print("dataset zip:", DATASET_ZIP, "->", os.path.isfile(DATASET_ZIP))
print("repo zip   :", REPO_ZIP, "->", os.path.isfile(REPO_ZIP), "| REPO_URL:", REPO_URL or "(not set)")

dataset zip: /content/drive/MyDrive/traffic-sign-inventory/datasets/arts_yolo.zip -> True
repo zip   : /content/drive/MyDrive/traffic-sign-inventory/repo_for_colab.zip -> True | REPO_URL: https://github.com/JawadGigyani/Geo-Localization-of-Traffic-Signs.git


In [3]:
# A3. Get the code
import os, shutil, subprocess, zipfile

shutil.rmtree(REPO, ignore_errors=True)
if REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO], check=True)
else:
    assert os.path.isfile(REPO_ZIP), f"Upload the repository zip to {REPO_ZIP}, or set REPO_URL in A2"
    with zipfile.ZipFile(REPO_ZIP) as z:
        z.extractall(REPO)
os.chdir(REPO)
assert os.path.isfile("pipeline_core.py"), "The zip should contain the repository files at its top level"
print(sorted(os.listdir(REPO)))

['.env.example', '.git', '.github', '.gitignore', 'LICENSE', 'README.md', 'backend', 'data', 'docs', 'frontend', 'notebooks', 'pipeline_core.py', 'requirements.txt', 'results', 'scripts', 'tests', 'weights']


In [4]:
# A4. Install the pinned detector library (Colab already has PyTorch), run the unit tests
!pip -q install ultralytics==8.4.120 python-dotenv
import platform, torch, ultralytics
print("python", platform.python_version(), "| torch", torch.__version__,
      "| ultralytics", ultralytics.__version__, "| GPU:", torch.cuda.is_available())
!python -m unittest discover -s tests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 8.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
python 3.13.15 | torch 2.11.0+cu130 | ultralytics 8.4.120 | GPU: True
......................
----------------------------------------------------------------------
Ran 22 tests in 0.005s

OK


## B. Data

Copies `arts_yolo.zip` to Colab's local disk and unpacks it into `data/arts_yolo/`
(3 minutes on the recorded run), then rebuilds the 30 held-out road sequences exactly the
way notebook 01, cell F1 did. Expect `TOTAL: 30 sequences 1108 frames 609 distinct signs`.

In [5]:
# B1. Restore the converted dataset
import os, time
os.makedirs(f"{REPO}/data", exist_ok=True)
if not os.path.isfile(f"{REPO}/data/arts_yolo/meta/frames_meta.json"):
    t0 = time.time()
    !cp "{DATASET_ZIP}" /content/arts_yolo.zip
    !unzip -q -o /content/arts_yolo.zip -d "{REPO}/data"
    !rm -f /content/arts_yolo.zip
    print(f"restored in {(time.time() - t0) / 60:.1f} min")
!ls "{REPO}/data/arts_yolo"

restored in 3.4 min
classes.txt  data.yaml	demo_sequence  images  labels  meta


In [6]:
# B2. Rebuild the held-out sequences and the split manifest
!python scripts/05_export_sequences.py

converted dataset: /content/repo/data/arts_yolo  (15135 photos)
split manifest -> /content/repo/results/split_manifest.csv  {'test': 2502, 'train': 10332, 'val': 2301}
leakage check -> /content/repo/results/split_check.json  ids shared {'train_val': 0, 'train_test': 0, 'val_test': 0}  posts shared {'train_val': 0, 'train_test': 1, 'val_test': 0}
test tiles with >= 20 photos: 42  (exporting 30)
  seq_8852_-14516        86 frames  150 distinct signs
  seq_8795_-14425        74 frames   15 distinct signs
  seq_8849_-14510        69 frames   35 distinct signs
  seq_8839_-14500        68 frames   32 distinct signs
  seq_8578_-14511        56 frames   16 distinct signs
  seq_8987_-14442        44 frames   62 distinct signs
  seq_8895_-14640        42 frames   12 distinct signs
  seq_8884_-14403        40 frames   38 distinct signs
  seq_8852_-14517        39 frames   28 distinct signs
  seq_8593_-14503        35 frames    9 distinct signs
  seq_8884_-14404        34 frames   27 distinct sign

## C. Reproduce the scores

C1 re-measures the two models. C2 and C3 run the whole pipeline on the 18
evaluation sequences, with the camera constants fitted on the 12 calibration
sequences. Section E compares everything with the expected values.

In [7]:
# C1. Detector mAP on the test split, classifier top-1 on validation crops
!python scripts/08_evaluate_models.py

DETECTOR on the test split
Ultralytics 8.4.120 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,591,902 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3423.8±679.4 MB/s, size: 427.6 KB)
val: Scanning /content/repo/data/arts_yolo/labels/test... 2502 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2502/2502 1.3Kit/s 1.9s
val: New cache created: /content/repo/data/arts_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 157/157 2.4it/s 1:05
                   all       2502       4518      0.739      0.677      0.723      0.495
                 D10-2        332        332      0.951      0.985      0.991      0.622
                 W11-2        253        257      0.871      0.897      0.928      0.754
                WT-100        119        187      0.862      0.909      0.936      0.652
                 W16-7    

In [8]:
# C2. The pipeline, with the crop classifier
!python scripts/07_batch_evaluate.py

sequences found: 30
weights: /content/repo/weights/best.pt   imgsz: 1280
second stage: crop_classifier.pt, 11 confusable families, overrules at >= 0.8

DETECTION
  seq_8570_-14512                30 frames    90 detections   24 ids ->  20 posts  [detected]
  seq_8578_-14511                56 frames   130 detections   16 ids ->  16 posts  [detected]
  seq_8593_-14503                35 frames    58 detections    9 ids ->   7 posts  [detected]
  seq_8726_-14468                28 frames    73 detections    5 ids ->   5 posts  [detected]
  seq_8730_-14467                33 frames    55 detections    7 ids ->   7 posts  [detected]
  seq_8731_-14463                23 frames    42 detections   12 ids ->   9 posts  [detected]
  seq_8749_-14452                33 frames    31 detections    6 ids ->   6 posts  [detected]
  seq_8762_-14443                24 frames    27 detections    4 ids ->   4 posts  [detected]
  seq_8795_-14425                74 frames   129 detections   15 ids ->  15 posts  [de

In [9]:
# C3. The pipeline, detector labels only (measures what the classifier adds)
!CLS_WEIGHTS=none python scripts/07_batch_evaluate.py

sequences found: 30
weights: /content/repo/weights/best.pt   imgsz: 1280
second stage: disabled

DETECTION
  seq_8570_-14512                30 frames    90 detections   24 ids ->  20 posts  [detected]
  seq_8578_-14511                56 frames   130 detections   16 ids ->  16 posts  [detected]
  seq_8593_-14503                35 frames    58 detections    9 ids ->   7 posts  [detected]
  seq_8726_-14468                28 frames    73 detections    5 ids ->   5 posts  [detected]
  seq_8730_-14467                33 frames    55 detections    7 ids ->   7 posts  [detected]
  seq_8731_-14463                23 frames    42 detections   12 ids ->   9 posts  [detected]
  seq_8749_-14452                33 frames    31 detections    6 ids ->   6 posts  [detected]
  seq_8762_-14443                24 frames    27 detections    4 ids ->   4 posts  [detected]
  seq_8795_-14425                74 frames   129 detections   15 ids ->  15 posts  [detected]
  seq_8833_-14417                25 frames    4

## D. Optional checks

The single-folder tools on the demo sequence and the video-tracker baseline. D3
measures the baselines and sanity checks quoted in the README (photo spacing,
BoT-SORT and ByteTrack, the single-sequence camera fit, error along and across the
line of sight, duplicate ids) and writes them to `results/sequence_checks.json`.

In [10]:
# D1. One folder: place signs with the stored constants, then score them
!SKIP_UPLOAD=1 python scripts/03_run_pipeline.py
!python scripts/06_evaluate_geo_and_dedupe.py

SEQUENCE_DIR = /content/repo/data/demo_sequence
WEIGHTS_PATH = /content/repo/weights/best.pt
Frames: 69
frames_meta.json: loaded
IMGSZ        = 1280 (must match training resolution)
ASSOCIATION  = geometric
Second stage: crop_classifier.pt over 11 confusable families
Raw detections: 103
  stage 2 changed 8/103 labels (7.8%)
Geometry: hfov=80 deg  size_k=0.881 m  radius=60 m  (from calibration.json)
Clustered into 22 physical signs (18 seen in more than one frame)
Unique tracks kept: 22
GPS sources: {'geo_projected': 22}
Local result -> /content/repo/data/pipeline_result_demo_sequence.json
SKIP_UPLOAD=1 -- not writing to Supabase.
Loaded 22 rows from pipeline_result_demo_sequence.json
Ground-truth frames: 69

=== 1. Geo-localization error (map pin vs true sign GPS) ===
scored rows       : 21
mean error        : 3.9 m
median error      : 3.0 m
90th percentile   : 9.0 m
worst             : 10.6 m

  median error, camera pin (no projection) : 7.3 m
  median error, geometric projection     

In [11]:
# D2. The BoT-SORT baseline (add TRACKER=bytetrack.yaml for ByteTrack)
!ASSOCIATION=tracker SKIP_UPLOAD=1 python scripts/03_run_pipeline.py

SEQUENCE_DIR = /content/repo/data/demo_sequence
WEIGHTS_PATH = /content/repo/weights/best.pt
Frames: 69
frames_meta.json: loaded
IMGSZ        = 1280 (must match training resolution)
ASSOCIATION  = tracker
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 384ms
Prepared 1 package in 489ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Tracker botsort.yaml: 8 of 103 detections were given an identity
Unique tracks kept: 8
GPS sources: {'frames_meta': 8}
Local result -> /content/repo/data/pipeline_result_demo_sequence.json
SKIP_UPLOAD=1 -- not writing to Supabase.


In [12]:
# D3. Baselines and sanity checks on the demo sequence
!python scripts/11_sequence_checks.py

sequence: demo_sequence (69 photos)
1. photo spacing: median 8.08 m over 68 gaps
3. camera fit: stored 80 deg / 0.881 m; this sequence alone 95 deg / 0.882 m (98 matches)
4. position error over 21 rows: median 3.02 m (along the line of sight 2.26 m, across 1.07 m)
2. botsort              8 of 103 detections given an identity
2. bytetrack            7 of 103 detections given an identity
2. position_clustering  103 of 103 detections given an identity
5. 35 ids -> 32 posts; 3 merged ids, max distance 0.0 m, 2 appear in exactly the same photos

saved -> /content/repo/results/sequence_checks.json


## E. Compare and save

In [13]:
# E1. Reproduced versus expected
import json, os

def load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

m   = load("results/model_metrics.json")
s2  = load("results/batch_evaluation_stage2.json")
s1  = load("results/batch_evaluation_stage1.json")
cal = load("weights/calibration.json")
det, clf = m["detector_test"], m["classifier_val"]
geo, c2 = s2["geo_error_m"], s2["classification"]
conf = c2.get("confusable") or {}

# name, expected (README), reproduced, tolerance
checks = [
    ("detector mAP50 (test split)",    0.723, det["map50"],                   0.01),
    ("detector mAP50-95 (test split)", 0.495, det["map50_95"],                0.01),
    ("detector precision",             0.739, det["precision"],               0.01),
    ("detector recall",                0.677, det["recall"],                  0.01),
    ("classifier top-1 (val crops)",   0.966, clf["top1"],                    0.01),
    ("classifier validation crops",    2173,  clf["crops"],                   0),
    ("calibration field of view",      80,    cal["hfov_deg"],                0),
    ("calibration size_k (m)",         0.88,  cal["size_k_m"],                0.01),
    ("calibration matched detections", 771,   cal.get("matched_detections"),  0),
    ("held-out posts",                 366,   s2["pooled"]["posts"],          0),
    ("median position error (m)",      4.7,   geo["median"],                  0.2),
    ("p90 position error (m)",         17.2,  geo["p90"],                     1.0),
    ("raw GPS baseline, median (m)",   10.7,  geo["baseline_median"],         0.3),
    ("look-alike accuracy, stage 1",   0.787, conf.get("stage1_acc"),         0.01),
    ("look-alike accuracy, stage 2",   0.943, conf.get("stage2_acc"),         0.01),
    ("all detections, stage 2",        0.948, c2.get("stage2_acc"),           0.01),
    ("signposts found (recall)",       0.877, s2["pooled"]["recall"],         0.01),
    ("deduplication, stage 2",         0.679, s2["pooled"]["dedupe"],         0.01),
    ("deduplication, stage 1",         0.495, s1["pooled"]["dedupe"],         0.01),
]
if os.path.isfile("results/sequence_checks.json"):
    sq = load("results/sequence_checks.json")
    ab = sq["association_baselines"]
    checks += [
        ("demo: photo spacing, median (m)", 8.08, sq["photo_spacing_m"]["median"],          0.1),
        ("demo: BoT-SORT identities",       8,    ab["botsort"]["with_identity"],           0),
        ("demo: ByteTrack identities",      7,    ab["bytetrack"]["with_identity"],         0),
        ("demo: single-sequence fit (deg)", 95,   sq["camera_fit"]["this_sequence_in_sample"]["hfov_deg"], 0),
    ]
if os.path.isfile("results/split_check.json"):
    sc = load("results/split_check.json")
    checks.append(("sign ids shared train/test", 0, sc["ids_shared"]["train_test"], 0))
    print("posts shared between splits (same class within 2 m):", sc["posts_shared"])

print(f"{'quantity':34s} {'expected':>9s} {'reproduced':>11s}  verdict")
for name, expected, got, tol in checks:
    if got is None:
        verdict = "missing"
    else:
        verdict = "OK" if abs(got - expected) <= tol else "CHECK"
    print(f"{name:34s} {expected:>9} {got:>11}  {verdict}")
print("\nversions:", s2["versions"])

posts shared between splits (same class within 2 m): {'train_val': 0, 'train_test': 1, 'val_test': 0}
quantity                            expected  reproduced  verdict
detector mAP50 (test split)            0.723      0.7229  OK
detector mAP50-95 (test split)         0.495      0.4946  OK
detector precision                     0.739      0.7391  OK
detector recall                        0.677      0.6773  OK
classifier top-1 (val crops)           0.966      0.9659  OK
classifier validation crops             2173        2173  OK
calibration field of view                 80        80.0  OK
calibration size_k (m)                  0.88      0.8807  OK
calibration matched detections           771         771  OK
held-out posts                           366         366  OK
median position error (m)                4.7      4.7337  OK
p90 position error (m)                  17.2     17.1979  OK
raw GPS baseline, median (m)            10.7     10.6753  OK
look-alike accuracy, stage 1           

In [14]:
# E2. Save the result files to Drive (download this zip and unpack it at the repository root)
import os, zipfile
os.makedirs(os.path.dirname(RESULTS_ZIP), exist_ok=True)
with zipfile.ZipFile(RESULTS_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk("results"):
        for name in sorted(files):
            z.write(os.path.join(root, name))
    z.write("weights/calibration.json")
    for name in z.namelist():
        print("  ", name)
print("saved ->", RESULTS_ZIP, f"({os.path.getsize(RESULTS_ZIP) // 1024} KB)")

   results/batch_evaluation_stage1.json
   results/batch_evaluation_stage2.json
   results/model_metrics.json
   results/sequence_checks.json
   results/sequences_manifest.json
   results/split_check.json
   results/split_manifest.csv
   weights/calibration.json
saved -> /content/drive/MyDrive/traffic-sign-inventory/export/reproduced_results.zip (85 KB)


## F. Optional: fill the dashboard from Colab

Uploads the 18 evaluation sequences (327 rows) to the Supabase project, so the
dashboard has data without any photographs on the local machine.

Before running it:
1. Create the Supabase table and the two storage buckets (README, Quick start steps 3 and 4).
2. In Colab's **Secrets** panel (the key icon on the left), add two secrets named exactly
   `SUPABASE_URL` and `SUPABASE_SERVICE_ROLE_KEY`, and switch on **Notebook access** for both.
3. This runtime must still have the sequences from B2. After a disconnect, re-run
   A1 to A4 and B1 to B2 first (section C is not needed).

The keys are written only to this Colab machine's `.env`, never to Drive. Run F1
once: running it again inserts the rows a second time.


In [16]:
# F1. Upload the evaluation sequences to Supabase
import os
from google.colab import userdata
!pip -q install supabase

values = {}
for name in ("SUPABASE_URL", "SUPABASE_SERVICE_ROLE_KEY"):
    try:
        values[name] = userdata.get(name)
    except Exception as e:
        raise SystemExit(f"Secret {name} is missing or not shared with this notebook ({type(e).__name__}). "
                         "Add it in the Secrets panel (key icon) and switch on 'Notebook access'.")
assert os.path.isdir("data/sequences"), "No sequences on this runtime: run A1-A4 and B1-B2 first"

with open(".env", "w") as f:
    for name, value in values.items():
        f.write(f"{name}={value}\n")
!UPLOAD=1 python scripts/07_batch_evaluate.py


sequences found: 30
weights: /content/repo/weights/best.pt   imgsz: 1280
second stage: crop_classifier.pt, 11 confusable families, overrules at >= 0.8

DETECTION
  cache: detections_cache_conf0.25_stage2.json (30 sequences)
  seq_8570_-14512                30 frames    90 detections   24 ids ->  20 posts  [cached]
  seq_8578_-14511                56 frames   130 detections   16 ids ->  16 posts  [cached]
  seq_8593_-14503                35 frames    58 detections    9 ids ->   7 posts  [cached]
  seq_8726_-14468                28 frames    73 detections    5 ids ->   5 posts  [cached]
  seq_8730_-14467                33 frames    55 detections    7 ids ->   7 posts  [cached]
  seq_8731_-14463                23 frames    42 detections   12 ids ->   9 posts  [cached]
  seq_8749_-14452                33 frames    31 detections    6 ids ->   6 posts  [cached]
  seq_8762_-14443                24 frames    27 detections    4 ids ->   4 posts  [cached]
  seq_8795_-14425                74 fram